RANDOM FOREST

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

In [ ]:
CSV_INPUT_PATH = "audio_dataset.csv"
LABEL_COLUMN="portata"
MAX_N_ESTIMATORS=1000

In [ ]:
# 1. CARICAMENTO DATI
# Assumiamo che il file CSV abbia la colonna identificativa come prima colonna
df = pd.read_csv(CSV_INPUT_PATH, index_col=0)

In [ ]:
# 2. PREPARAZIONE X e y
target_column = 'portata'
X = df.drop(columns=[target_column])
y = df[target_column]

# 3. SPLIT TRAIN/TEST
# Dividiamo i dati: 80% training, 20% test
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Seconda separazione: dal rimanente 80%, prendiamo il Validation Set (es. 25% di 80% = 20% del totale)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

# Risultato: Train 60%, Val 20%, Test 20%

EARLY STOPPING

In [ ]:
'''
rf = RandomForestRegressor(n_estimators=10, warm_start=True, random_state=42, n_jobs=-1)

prev_mae = np.inf
patience = 5
count_no_improvement = 0
best_n_estimators = 10

for n in range(20,  MAX_N_ESTIMATORS + 1, 10):
    rf.n_estimators = n
    rf.fit(X_train, y_train)
    
    # Predizione sul set di validazione
    preds = rf.predict(X_val)
    current_mae = mean_absolute_error(y_val, preds)
    
    print(f"Alberi: {n} | MAE: {current_mae:.5f}")
    
    # Nota: qui cerchiamo una DIMINUZIONE del MAE
    if current_mae < prev_mae - 0.0001:
        prev_mae = current_mae
        best_n_estimators = n
        count_no_improvement = 0
    else:
        count_no_improvement += 1
        
    if count_no_improvement >= patience:
        break

print(f"Numero ottimale di alberi individuato: {best_n_estimators}")
'''

OTTIMIZZAZIONE IPERPARAMETRI

In [ ]:
'''
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_features': ['sqrt', 'log2', 0.3, 0.5],
    'max_depth': [None, 10, 20, 30],
    'min_samples_leaf': [1, 2, 4, 10],
    'bootstrap': [True]
}

rf = RandomForestRegressor(n_jobs=-1, random_state=42)

# Prova 20 combinazioni diverse casualmente
random_search = RandomizedSearchCV(rf, param_distributions=param_dist, 
                                   n_iter=20, cv=5, scoring='neg_mean_absolute_error')

random_search.fit(X_train, y_train)

print(f"Migliori parametri: {random_search.best_params_}")
'''

In [ ]:
# Definiamo la griglia dei parametri
# Nota: n_estimators lo teniamo fisso a un valore alto per ottimizzare gli altri, 
# oppure ne testiamo solo due valori per non esplodere con le combinazioni.
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 15, 30],
    'max_features': ['sqrt', "log2", 0.3, 0.5], # 'sqrt' è circa il 10-15% se hai molte feature
    'min_samples_leaf': [1, 2, 5],
    'min_samples_split': [2, 5]
}

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

# Configuriamo la ricerca
grid_search = GridSearchCV(
    estimator=rf, 
    param_grid=param_grid, 
    cv=5,                         # 3-fold cross-validation per velocità, 5 per precisione
    scoring='neg_mean_absolute_error', 
    verbose=1,                    # Per vedere il progresso in tempo reale
    n_jobs=-1                     # Usa tutti i core del processore
)

# Eseguiamo la ricerca
grid_search.fit(X_train, y_train)

# Risultati
print("Migliori parametri individuati:")
print(grid_search.best_params_)

DEFINIZIONE FINALE DEL MODELLO

In [ ]:
'''
# 4. TRAINING RANDOM FOREST REGRESSOR
# Usiamo RandomForestRegressor per prevedere valori continui
rf_regressor = RandomForestRegressor(n_estimators=best_n_estimators, random_state=42, n_jobs=-1)
rf_regressor.fit(X_train, y_train)
print("Training completato.")
'''
# Il miglior modello è già pronto all'uso:
rf_regressor = grid_search.best_estimator_

VALUTAZIONE MODELLO FINALE

In [ ]:
# 5. PREDIZIONE E VALUTAZIONE
y_pred = rf_regressor.predict(X_test)

In [ ]:
# Metriche di Regressione
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"--- PERFORMANCE RANDOM FOREST ---")
print(f"R^2 Score: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

In [ ]:
# VISUALIZZAZIONE: FEATURE IMPORTANCE
plt.figure(figsize=(10, 6))
importances = pd.Series(rf_regressor.feature_importances_, index=X.columns)
importances.nlargest(10).sort_values(ascending=True).plot(kind='barh', color='orange')
plt.title("Top 10 Feature per la stima della Portata")
plt.xlabel("Importanza Relativa")
plt.tight_layout()
plt.show()

In [ ]:
# SALVATAGGIO MODELLO (Opzionale)
# import joblib
# joblib.dump(rf_model, 'random_forest_audio.pkl')